In [1]:
# ==========================================================
# CFN · INFERENCIA SATLAS (GPU si existe, si no CPU)
# Rutas fijas que tú pediste (2016)
# ==========================================================

# 0) Montar Drive y librerías
from google.colab import drive
drive.mount('/content/drive')

!pip -q install rasterio tifffile pillow opencv-python
!pip -q install torch torchvision --extra-index-url https://download.pytorch.org/whl/cu121
!pip -q install satlaspretrain-models

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ==========================================================
# CFN · INFERENCIA SATLAS (optimizada + barra de progreso)
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

!pip -q install rasterio tqdm torch torchvision --extra-index-url https://download.pytorch.org/whl/cu121
!pip -q install satlaspretrain-models pillow opencv-python

# --------------------------
# 1. Imports y rutas
# --------------------------
import os, math, numpy as np, torch, rasterio, gc
from rasterio.windows import Window
from tqdm import tqdm
import satlaspretrain_models as spm

BIG_IMG  = "/content/drive/MyDrive/5_PROYECTOS/CFN/1_Ponencia/Procesamiento/T4/data/Sentinel_RGBNIR_2016_CAR.tif"
BASE_OUT = "/content/drive/MyDrive/5_PROYECTOS/CFN/1_Ponencia/Procesamiento/T4/data/output/2016"
FULL_OUT = os.path.join(BASE_OUT, "pred_full_merged_2016.tif")
os.makedirs(BASE_OUT, exist_ok=True)

WEIGHTS  = "/content/drive/MyDrive/5_PROYECTOS/CFN/1_Ponencia/Procesamiento/T4/data/finetune/best_finetuned.pth"

print("🗺️  Imagen:", BIG_IMG)
print("📂 Salida :", BASE_OUT)
print("🗃️  OUT   :", FULL_OUT)

# --------------------------
# 2. Dispositivo
# --------------------------
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    device = torch.device("cuda")
    use_amp = True
    print(f"✅ GPU detectada: {gpu_name} (AMP activado)")
else:
    device = torch.device("cpu")
    use_amp = False
    print("⚠️ No hay GPU disponible. Se usará CPU.")

torch.set_grad_enabled(False)

# --------------------------
# 3. Parámetros
# --------------------------
TILE, STRIDE = 512, 512
NDVI_THR     = 0.19
PROB_THR     = 0.40
PERC_RESCALE = True

# --------------------------
# 4. Funciones auxiliares
# --------------------------
def _prep_rgb_from_bands(B2, B3, B4, perc=True):
    rgb = np.stack([B4, B3, B2], axis=0)
    if perc:
        arr = rgb.copy()
        for c in range(3):
            p2, p98 = np.percentile(arr[c], [2, 98])
            if p98 > p2:
                arr[c] = np.clip((arr[c]-p2)/(p98-p2), 0, 1)
        rgb = arr
    return np.transpose(rgb, (1, 2, 0))

def _scale_guess(*bands):
    mx = max(float(b.max()) for b in bands if b.size)
    return 10000.0 if mx > 5000 else (255.0 if mx > 1.5 else 1.0)

# --------------------------
# 5. Cargar modelo y pesos
# --------------------------
_real_torch_load = torch.load
def _torch_load_cpu(*args, **kwargs):
    kwargs["map_location"] = "cpu"
    return _real_torch_load(*args, **kwargs)
torch.load = _torch_load_cpu

weights_manager = spm.Weights()
model = weights_manager.get_pretrained_model(
    model_identifier="Sentinel2_SwinB_SI_RGB",
    fpn=True, head=spm.Head.SEGMENT, num_categories=2
).to(device).eval()

torch.load = _real_torch_load
ckpt = torch.load(WEIGHTS, map_location="cpu")
state = ckpt.get("model", ckpt)
missing, unexpected = model.load_state_dict(state, strict=False)
print("🔧 Pesos cargados. Missing:", missing, "| Unexpected:", unexpected)
model.to(device).eval()

# --------------------------
# 6. Inferencia por tiles (streaming)
# --------------------------
with rasterio.open(BIG_IMG) as src:
    H, W = src.height, src.width
    transform, crs = src.transform, src.crs
    profile = src.profile.copy()
    profile.update({"count": 1, "dtype": rasterio.uint8, "compress": "LZW"})

    n_tiles_x = math.ceil(W / STRIDE)
    n_tiles_y = math.ceil(H / STRIDE)
    total = n_tiles_x * n_tiles_y

    with rasterio.open(FULL_OUT, "w", **profile) as dst:
        pbar = tqdm(total=total, desc="Tiles procesados", ncols=80)
        for ty in range(0, H, STRIDE):
            for tx in range(0, W, STRIDE):
                ph = min(TILE, H - ty)
                pw = min(TILE, W - tx)
                win = Window(tx, ty, pw, ph)

                # Leer bandas
                B2 = src.read(1, window=win).astype(np.float32, copy=False)
                B3 = src.read(2, window=win).astype(np.float32, copy=False)
                B4 = src.read(3, window=win).astype(np.float32, copy=False)
                B8 = src.read(4, window=win).astype(np.float32, copy=False)

                # Normalizar
                scale = _scale_guess(B2, B3, B4, B8)
                B2 /= scale; B3 /= scale; B4 /= scale; B8 /= scale

                # NDVI candidatos
                ndvi = (B8 - B4) / (B8 + B4 + 1e-6)
                ndvi = np.clip(ndvi, -1, 1)
                cand = (ndvi >= NDVI_THR)

                if not cand.any():
                    dst.write(np.zeros((ph, pw), dtype=np.uint8), 1, window=win)
                    pbar.update(1)
                    continue

                rgb = _prep_rgb_from_bands(B2, B3, B4, perc=PERC_RESCALE)

                # Padding
                need_pad = (ph < TILE) or (pw < TILE)
                if need_pad:
                    pad_h, pad_w = TILE - ph, TILE - pw
                    rgb_pad  = np.pad(rgb,  ((0,pad_h),(0,pad_w),(0,0)), mode="reflect")
                    cand_pad = np.pad(cand, ((0,pad_h),(0,pad_w)),   mode="constant", constant_values=False)
                else:
                    rgb_pad, cand_pad = rgb, cand

                t = torch.from_numpy(rgb_pad.transpose(2,0,1)).unsqueeze(0).float().to(device, non_blocking=True)

                if use_amp:
                    scaler_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
                else:
                    class _nullctx:
                        def __enter__(self): return None
                        def __exit__(self, *a): return False
                    scaler_ctx = _nullctx()

                with scaler_ctx:
                    out = model(t)
                    if isinstance(out,(tuple,list)): out = out[0]
                    if isinstance(out,dict):
                        out = out.get('out', out.get('logits',
                               next(v for v in out.values() if torch.is_tensor(v))))
                    prob_pad = torch.softmax(out, dim=1)[0,1].detach().cpu().numpy()

                prob = prob_pad[:ph, :pw] if need_pad else prob_pad
                cand_local = cand_pad[:ph, :pw] if need_pad else cand_pad
                mask_local = ((prob >= PROB_THR) & cand_local).astype(np.uint8)
                dst.write(mask_local, 1, window=win)

                # liberar memoria
                del B2,B3,B4,B8,ndvi,cand,rgb,t,out,prob_pad,prob,cand_local,mask_local
                if use_amp and torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()

                pbar.update(1)

        pbar.close()

print("✅ Inferencia completada.")
print("🗃️  Guardado:", FULL_OUT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🗺️  Imagen: /content/drive/MyDrive/5_PROYECTOS/CFN/1_Ponencia/Procesamiento/T4/data/Sentinel_RGBNIR_2016_CAR.tif
📂 Salida : /content/drive/MyDrive/5_PROYECTOS/CFN/1_Ponencia/Procesamiento/T4/data/output/2016
🗃️  OUT   : /content/drive/MyDrive/5_PROYECTOS/CFN/1_Ponencia/Procesamiento/T4/data/output/2016/pred_full_merged_2016.tif
✅ GPU detectada: Tesla T4 (AMP activado)
🔧 Pesos cargados. Missing: [] | Unexpected: []


Tiles procesados: 100%|█████████████████████| 2223/2223 [10:29<00:00,  3.53it/s]


✅ Inferencia completada.
🗃️  Guardado: /content/drive/MyDrive/5_PROYECTOS/CFN/1_Ponencia/Procesamiento/T4/data/output/2016/pred_full_merged_2016.tif
